# 00 - Parámetros Globales del Proyecto

## 1. Descripción

Este notebook se corre primero en cualquier notebook de la fase de Modelado (mediante `%run ./00_parametros_globales.ipynb`). Fija las semillas de reproducibilidad, detecta si hay GPU NVIDIA disponible, define el flag `SAMPLING_MODE` (para iterar rápido en una laptop de capacidad media antes de escalar a un servidor con GPU) y resuelve las rutas del proyecto de forma relativa (funciona tanto si el notebook se abre desde `notebooks/` como si el proyecto se mueve de carpeta).

**Contenido de este notebook:**

1. Descripción
2. Importaciones
3. Configuración
   - 3.1 Reproducibilidad
   - 3.2 Hardware (detección de GPU)
   - 3.3 Modo de muestreo
   - 3.4 Ventana temporal del CNN-LSTM
   - 3.5 Métrica de optimización
   - 3.6 Rutas del proyecto
4. Resultados (verificación final de la configuración)


## 2. Importaciones

In [1]:
# Librerías estándar para reproducibilidad y manejo de rutas, más numpy/torch
# para fijar semillas en todos los generadores aleatorios relevantes del proyecto.
import random
from pathlib import Path

import numpy as np
import torch


## 3. Configuración

### 3.1 Reproducibilidad

Se fija una única semilla (`RANDOM_STATE = 42`) en Python, NumPy y PyTorch, para que cualquier resultado del proyecto (splits, SMOTENC, inicialización de pesos) sea reproducible entre corridas.

In [2]:
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)


### 3.2 Hardware: detección automática de GPU NVIDIA

Se detecta automáticamente si hay una GPU NVIDIA disponible (`torch.cuda.is_available()`). El entrenamiento del CNN-LSTM y TabNet usa `DEVICE` para moverse a GPU cuando existe, sin necesidad de configuración manual al pasar de la laptop local al servidor de producción.

In [3]:
USE_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_GPU else "cpu")

print(f"GPU NVIDIA detectada: {USE_GPU}")
if USE_GPU:
    print(f"  Dispositivo: {torch.cuda.get_device_name(0)}")
else:
    print("  Corriendo en CPU. Se recomienda SAMPLING_MODE=True para iterar más rápido.")


GPU NVIDIA detectada: False
  Corriendo en CPU. Se recomienda SAMPLING_MODE=True para iterar más rápido.


### 3.3 Modo de muestreo

Con `SAMPLING_MODE=True`, los notebooks de modelado deben tomar una muestra reducida y ESTRATIFICADA (por año y por clase del target) del dataset, en vez de usar `dataset_modelado_personas.csv` completo. Esto permite iterar rápido en una laptop de capacidad media; se cambia a `False` para las corridas finales, idealmente en el servidor con GPU.

In [4]:
SAMPLING_MODE = False  # True para iterar rápido en laptop; False para corridas finales en servidor con GPU
FRACCION_MUESTRA = 0.2  # 20% de las filas si SAMPLING_MODE=True


### 3.4 Ventana temporal del CNN-LSTM

Decidida en el plan de modelado: 3 años hacia atrás por persona, con máscara para encuestados de años iniciales que no tienen historia completa.

In [5]:
VENTANA_TEMPORAL_ANIOS = 3


### 3.5 Métrica de optimización

PR-AUC es la métrica reina para el ajuste de hiperparámetros y la selección de modelos (más robusta que Accuracy bajo desbalance de clases); F1 de la clase "Satisfecho" se reporta como métrica secundaria.

In [6]:
METRICA_OPTIMIZACION = "pr_auc"  # PR-AUC como criterio principal; F1 de "Satisfecho" como métrica secundaria de reporte


### 3.6 Rutas del proyecto

Las rutas se resuelven de forma relativa a partir del directorio de trabajo actual, así que funcionan igual si el notebook se abre desde `notebooks/` o si el proyecto entero se mueve de carpeta.

In [7]:
RAIZ_PROYECTO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RUTA_DATA_RAW = RAIZ_PROYECTO / "data" / "raw"
RUTA_DATA_PROCESSED = RAIZ_PROYECTO / "data" / "processed"
RUTA_DATA_MODELS = RAIZ_PROYECTO / "data" / "models"
RUTA_REPORTS = RAIZ_PROYECTO / "reports"
RUTA_FIGURES = RUTA_REPORTS / "figures"
RUTA_TABLAS = RUTA_REPORTS / "tablas"

RUTA_DATASET_PERSONAS = RUTA_DATA_PROCESSED / "dataset_modelado_personas.csv"
RUTA_PANEL_MACRO = RUTA_DATA_PROCESSED / "panel_macro_anual.csv"

# Crea las carpetas de salida si todavía no existen (data/raw y data/processed
# ya deberían existir porque son insumo, no salida, de este notebook).
for ruta in (RUTA_DATA_MODELS, RUTA_FIGURES, RUTA_TABLAS):
    ruta.mkdir(parents=True, exist_ok=True)


## 4. Resultados

Verificación final: confirma que la configuración quedó como se espera y que los datasets de entrada existen antes de continuar con cualquier otro notebook.

In [8]:
print(f"RAIZ_PROYECTO: {RAIZ_PROYECTO}")
print(f"SAMPLING_MODE: {SAMPLING_MODE} (fracción={FRACCION_MUESTRA if SAMPLING_MODE else 1.0})")
print(f"VENTANA_TEMPORAL_ANIOS: {VENTANA_TEMPORAL_ANIOS}")
print(f"RUTA_DATASET_PERSONAS existe: {RUTA_DATASET_PERSONAS.exists()}")
print(f"RUTA_PANEL_MACRO existe: {RUTA_PANEL_MACRO.exists()}")


RAIZ_PROYECTO: c:\TesisSD
SAMPLING_MODE: False (fracción=1.0)
VENTANA_TEMPORAL_ANIOS: 3
RUTA_DATASET_PERSONAS existe: True
RUTA_PANEL_MACRO existe: True
